# Evidencia de Aprendizaje 3 (EA3)\n\n**Taller: Procesamiento de Datos Económicos en Infraestructura Cloud**\n\nEste notebook está diseñado para ejecutarse en **Databricks Community Edition** usando el archivo `Gold_vs_Economic_Factors_2015_2026.csv` cargado en `/FileStore/tables/`.

## Instrucción 0: Diseño del esquema de datos\n\n### Diccionario de Datos\n\n| Entidad | Campo | Tipo de Dato | Llave | Nulabilidad | Descripción |\n| :--- | :--- | :--- | :--- | :--- | :--- |\n| **Factor** | `Date` | Date | PK | No | Fecha del registro económico. |\n| **Precio** | `Gold_Price_XAU_USD` | Double | - | No | Valor del oro en dólares. |\n| **Índice** | `US_Dollar_Index_DXY`| Double | - | No | Valor del dólar frente a divisas. |\n| **Energía**| `Crude_Oil_Price` | Double | - | No | Precio del petróleo crudo. |\n| **Riesgo** | `Inflation_Rate_Pct` | Double | - | No | Tasa de inflación porcentual. |

In [ ]:
from pyspark.sql.types import StructType, StructField, DoubleType, DateType\n\n# Diseño del esquema basado en la estructura del dataset CSV\nschema_economico = StructType([\n    StructField("Date", DateType(), False),\n    StructField("Gold_Price_XAU_USD", DoubleType(), False),\n    StructField("US_Dollar_Index_DXY", DoubleType(), False),\n    StructField("Crude_Oil_Price", DoubleType(), False),\n    StructField("Inflation_Rate_Pct", DoubleType(), False)\n])

### Diagrama de Entidad (Mermaid)\n\n```mermaid\nerDiagram\n    FACTORES_ECONOMICOS {\n        date Date PK\n        double Gold_Price_XAU_USD\n        double US_Dollar_Index_DXY\n        double Crude_Oil_Price\n        double Inflation_Rate_Pct\n    }\n```

## Instrucción 1: Configuración y evidencia de infraestructura

In [ ]:
# Verificación de versiones y configuración del Spark Context\nprint(f"Versión de Spark: {spark.version}")\nprint("Detalles del Clúster:")\nfor conf in spark.sparkContext.getConf().getAll():\n    print(conf)\n\n# Evidencia de la estructura de almacenamiento DBFS\ndisplay(dbutils.fs.ls("/FileStore/tables/"))

- **Databricks Runtime:** 10.4 LTS o superior.\n- **Capacidad:** Databricks CE ofrece aproximadamente 15.3 GB de RAM y 2 núcleos virtualizados.\n- **Almacenamiento:** Persistencia en **DBFS** (Databricks File System).

## Instrucción 2: Obtención e ingestión de datos

In [ ]:
# Carga del dataset aplicando el esquema diseñado (Instrucción 0)
ruta = "/FileStore/tables/Gold_vs_Economic_Factors_2015_2026.csv"

df_gold = spark.read.format("csv") \
    .option("header", "true") \
    .schema(schema_economico) \
    .load(ruta)

# Creación de tabla analítica permanente
df_gold.write.mode("overwrite").saveAsTable("factores_oro_analitica")

# Confirmación de recuento
print(f"Total de registros ingeridos: {df_gold.count()}")
display(df_gold.limit(5))

## Instrucción 3: Validaciones en Spark y SQL

In [ ]:
# A. Metadatos y descripción\ndf_gold.printSchema()

In [ ]:
%sql\nDESCRIBE factores_oro_analitica;

In [ ]:
# B. Agregación en Spark (filtro por inflación alta)
from pyspark.sql.functions import avg

umbral_inflacion_alta = 5

(df_gold.filter(f"Inflation_Rate_Pct > {umbral_inflacion_alta}")
       .select(avg("Gold_Price_XAU_USD"))
       .show())

In [ ]:
# Validación SQL usando el mismo umbral para consistencia
spark.sql(f"""
SELECT AVG(Gold_Price_XAU_USD) as Promedio_Precio_Oro
FROM factores_oro_analitica
WHERE Inflation_Rate_Pct > {umbral_inflacion_alta}
""").show()

## Instrucción 4: Comparación SQL vs. Spark\n\n| Característica | **Spark SQL** | **PySpark (Spark API)** |\n| :--- | :--- | :--- |\n| **Facilidad de uso** | Alta; lenguaje declarativo estándar. | Media; requiere lógica de programación funcional. |\n| **Transformaciones** | Limitado a estructuras relacionales. | Superior; permite pipelines complejos y UDFs. |\n| **Integración** | Ideal para herramientas de BI y analistas. | Ideal para Data Science y Machine Learning. |\n| **Rendimiento** | Optimizado por el catalizador de Spark. | Equivalente; ambos procesan en memoria. |\n\n**Conclusión:** SQL es excelente para la exploración rápida de datos estructurados, mientras que Spark es la herramienta definitiva para procesar billones de registros y ejecutar algoritmos de aprendizaje automático bajo el paradigma distribuido.